# Correr la imagen y levantar el contenedor en docker
- docker run -d --name mongo -p 27017:27017 mongo:8.0

# Crear un proyecto 
- uv init --no-package intro_mongodb --python 3.12

# Instalar dependencias
- uv add ipykernel pymongo

In [3]:
import pymongo
print(pymongo.__version__)

4.19.0


In [4]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
client.admin.command('ping')

{'ok': 1.0}

In [5]:
# Creando el schema -> base de datos

db = client['escuela']

In [6]:
# Creando la coleccion -> tabla
alumnos = db['alumnos']

In [7]:
# CRUD -> create / read / update / delete


# Insertando un documento a la coleccion -> un registro a la tabla
res = alumnos.insert_one(
    {
        'nombre':'Ana Torres',
        'edad': 21,
        'ciudad': 'Lima',
        'cursos': ['Python', 'SQL']
    }
)

print(res.inserted_id)

6adab40fbfbb91e2bb1653b0


In [9]:
# Insertar varios documentos a la colección, no siempre tiene todos los campos
# debe ser una lista de documentos (lista de diccionarios)

alumnos.insert_many(
    [
        {
            'nombre':'Luis Ramos',
            'edad': 19,
            'ciudad': 'Bogota'
            
        },
        {
            'nombre': 'Maria Paz',
            'edad': 23,
            'ciudad': 'Lima',
            'beca': True
        }
    ]
)

InsertManyResult([ObjectId('6adab4d1bfbb91e2bb1653b1'), ObjectId('6adab4d1bfbb91e2bb1653b2')], acknowledged=True)

In [ ]:

# .find() -> SELECT * FROM alumnos
list(alumnos.find())

[{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
  'nombre': 'Ana Torres',
  'edad': 21,
  'ciudad': 'Lima',
  'cursos': ['Python', 'SQL']},
 {'_id': ObjectId('6adab4d1bfbb91e2bb1653b1'),
  'nombre': 'Luis Ramos',
  'edad': 19,
  'ciudad': 'Bogota'},
 {'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'),
  'nombre': 'Maria Paz',
  'edad': 23,
  'ciudad': 'Lima',
  'beca': True}]

In [ ]:
# alumnos.find() -> SELECT * FROM alumnos

for alumno in alumnos.find():
    print(alumno)
    print('-'*50)

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
--------------------------------------------------
{'_id': ObjectId('6adab4d1bfbb91e2bb1653b1'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogota'}
--------------------------------------------------
{'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'), 'nombre': 'Maria Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
--------------------------------------------------


In [ ]:
# alumnos.find({'ciudad': 'Lima'}) -> SELECT * FROM alumnos WHERE ciudad='Lima'

for alumno in alumnos.find({'ciudad': 'Lima'}):
    print(alumno)
    print('-'*50)

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
--------------------------------------------------
{'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'), 'nombre': 'Maria Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
--------------------------------------------------


In [ ]:
# alumnos.find_one({'ciudad': 'Lima'}) -> SELECT * FROM alumnos WHERE ciudad='Lima' LIMIT 1
# como solo es uno, devuelve un diccionario

res = alumnos.find_one({'ciudad': 'Lima'})

for field in res:
    print(f"Key: {field} -> valor: {res[field]}")

Key: _id -> valor: 6adab40fbfbb91e2bb1653b0
Key: nombre -> valor: Ana Torres
Key: edad -> valor: 21
Key: ciudad -> valor: Lima
Key: cursos -> valor: ['Python', 'SQL']


In [ ]:
# alumnos.find({}, {'nombre':  1, 'edad': 1}) -> SELECT nombre, edad FROM alumnos 

for alumno in alumnos.find({}, {'nombre': 1, 'edad':1, '_id':0}):
    print(alumno)

{'nombre': 'Ana Torres', 'edad': 21}
{'nombre': 'Luis Ramos', 'edad': 19}
{'nombre': 'Maria Paz', 'edad': 23}


In [ ]:
# alumnos.find().sort({'edad': -1}) -> SELECT * FROM alumnos ORDER BY edad DESC
# 1: ascendente -1: descendente

for alumno in alumnos.find().sort({'edad': -1}):
    print(alumno)

{'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'), 'nombre': 'Maria Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
{'_id': ObjectId('6adab4d1bfbb91e2bb1653b1'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogota'}


In [20]:
# alumnos.count_documents({}) -> SELECT COUNT(1) FROM alumnos

alumnos.count_documents({})

3

In [21]:
# alumnos.count_documents({'edad': 21}) -> SELECT COUNT(1) FROM alumnos WHERE edad = 21

alumnos.count_documents({'edad': 21})

1

In [22]:
# alumnos.count_documents({'ciudad': 'Lima'}) -> SELECT COUNT(1) FROM alumnos WHERE ciudad = 'Lima'

alumnos.count_documents({'ciudad': 'Lima'})

2

In [23]:
# Adicionando un nuevo registro

res = alumnos.insert_one(
    {
        'nombre': 'Marisol Vasquez',
        'edad': 30,
        'direccion': {
            'ciudad': 'Lima',
            'distrito': 'Miraflores'
        },
        'cursos': ['Python', 'MongoDB']
    }
)

In [ ]:
# filtrar cuando un valor esta anidado

for alumno in alumnos.find({'direccion.distrito': 'Miraflores'}):
    print(alumno)

{'_id': ObjectId('6adab8a6bfbb91e2bb1653b3'), 'nombre': 'Marisol Vasquez', 'edad': 30, 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'}, 'cursos': ['Python', 'MongoDB']}


In [29]:
# igual, no igual
# MongoDB -> $eq $ne
# SQL -> = / <> / !=

for alumno in alumnos.find({
    'ciudad': {'$ne': 'Lima'}, 
    'direccion.ciudad': {'$ne': 'Lima'}
}):
    print(alumno)

{'_id': ObjectId('6adab4d1bfbb91e2bb1653b1'), 'nombre': 'Luis Ramos', 'edad': 19, 'ciudad': 'Bogota'}


In [ ]:
# gt -> mayor que >
# gte -> mayor o igual que >=
# lt -> menor que <
# lte -> menor o igual que <=

for alumno in alumnos.find({
    'edad': {'$gt': 19}
}):
    print(alumno)

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'), 'nombre': 'Ana Torres', 'edad': 21, 'ciudad': 'Lima', 'cursos': ['Python', 'SQL']}
{'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'), 'nombre': 'Maria Paz', 'edad': 23, 'ciudad': 'Lima', 'beca': True}
{'_id': ObjectId('6adab8a6bfbb91e2bb1653b3'), 'nombre': 'Marisol Vasquez', 'edad': 30, 'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'}, 'cursos': ['Python', 'MongoDB']}


# Update

In [32]:
# actualizar un registro
alumnos.update_one(
    {
        #primer valor, es el filtro
        'nombre':'Ana Torres'
    },
     {   #valor a actualizar
        "$set": {'ciudad':'Arequipa'} # fija o establece
    }
)

alumnos.find_one({'nombre':'Ana Torres'})

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
 'nombre': 'Ana Torres',
 'edad': 21,
 'ciudad': 'Arequipa',
 'cursos': ['Python', 'SQL']}

In [33]:
# actualizar varios registros
# ejemplo -> bajar un año a la edad
alumnos.update_many(
    {}, {"$inc": {'edad': -1}}
)

alumnos.find_one({'nombre':'Ana Torres'})

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
 'nombre': 'Ana Torres',
 'edad': 20,
 'ciudad': 'Arequipa',
 'cursos': ['Python', 'SQL']}

In [ ]:
# modificar arreglos, adicionar un elemento a un arreglo
alumnos.update_one(
    {
        #primer valor, es el filtro
        'nombre':'Ana Torres'
    },
     {   #valor a añadir
        "$push": {'cursos':'MongoDB'} # fija o establece
    }
)

alumnos.find_one({'nombre':'Ana Torres'})

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
 'nombre': 'Ana Torres',
 'edad': 20,
 'ciudad': 'Arequipa',
 'cursos': ['Python', 'SQL', 'MongoDB']}

In [35]:
# modificar arreglos
# quitarle un elemento a un arreglo

alumnos.update_one(
    {
        #primer valor, es el filtro
        'nombre':'Ana Torres'
    },
     {   #valor a añadir
        "$pull": {'cursos':'Python'} # fija o establece
    }
)

alumnos.find_one({'nombre':'Ana Torres'})

{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
 'nombre': 'Ana Torres',
 'edad': 20,
 'ciudad': 'Arequipa',
 'cursos': ['SQL', 'MongoDB']}

In [36]:
# Delete

list(alumnos.find())

[{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
  'nombre': 'Ana Torres',
  'edad': 20,
  'ciudad': 'Arequipa',
  'cursos': ['SQL', 'MongoDB']},
 {'_id': ObjectId('6adab4d1bfbb91e2bb1653b1'),
  'nombre': 'Luis Ramos',
  'edad': 18,
  'ciudad': 'Bogota'},
 {'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'),
  'nombre': 'Maria Paz',
  'edad': 22,
  'ciudad': 'Lima',
  'beca': True},
 {'_id': ObjectId('6adab8a6bfbb91e2bb1653b3'),
  'nombre': 'Marisol Vasquez',
  'edad': 29,
  'direccion': {'ciudad': 'Lima', 'distrito': 'Miraflores'},
  'cursos': ['Python', 'MongoDB']}]

In [38]:
from pymongo.collection import ObjectId
alumnos.delete_one(
    {'_id': ObjectId('6adab8a6bfbb91e2bb1653b3')}
)

list(alumnos.find())

[{'_id': ObjectId('6adab40fbfbb91e2bb1653b0'),
  'nombre': 'Ana Torres',
  'edad': 20,
  'ciudad': 'Arequipa',
  'cursos': ['SQL', 'MongoDB']},
 {'_id': ObjectId('6adab4d1bfbb91e2bb1653b1'),
  'nombre': 'Luis Ramos',
  'edad': 18,
  'ciudad': 'Bogota'},
 {'_id': ObjectId('6adab4d1bfbb91e2bb1653b2'),
  'nombre': 'Maria Paz',
  'edad': 22,
  'ciudad': 'Lima',
  'beca': True}]